# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = dataset.metadata.to_json()
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and their `@id`s, fields, and columns.

We enumerate all record sets in the dataset and print their fields and columns, referencing each by its `@id`.

In [ ]:
print("Available record sets:")
for record_set in dataset.metadata.record_sets:
    print(f"\nRecord set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    {field.id} (name: {field.name}, dataType: {field.data_type})")
    if hasattr(record_set, 'columns') and record_set.columns is not None:
        print("  Columns:")
        for column in record_set.columns:
            print(f"    {column.id} (name: {column.name}, dataType: {column.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Each record set and field is referenced by its `@id`.

This dataset contains one main record set representing the protein measurements. We'll extract all records from that set.

In [ ]:
# Get all record set @ids
record_sets = [rs.id for rs in dataset.metadata.record_sets]
print("Record set @ids available:", record_sets)

# For illustration, we load the first record set (typically the main table)
main_record_set_id = record_sets[0]

# Extract data from each record set into a dictionary of DataFrames
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

print(f"Fields (columns) available in the record set {main_record_set_id}:")
print(list(dataframes[main_record_set_id].columns))
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We identify a numeric field (e.g. 'cr:coverage_percent', representing the protein sequence coverage in percent), filter by a threshold, normalize, and group by another field (e.g. 'cr:description' as a category). All field/column selectors use their `@id` where available.

In [ ]:
# Select field @ids: update below after inspecting the output of cells above, if necessary
numeric_field = None
group_field = None

# This block finds a numeric field and a categorical field to work with using @id
df = dataframes[main_record_set_id]
for col in df.columns:
    if numeric_field is None and pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
    if group_field is None and pd.api.types.is_string_dtype(df[col]):
        group_field = col

print(f"Selected numeric field: {numeric_field}")
print(f"Selected group field: {group_field}")

# Set a threshold for filtering numeric values
threshold = df[numeric_field].mean() if numeric_field else 10
filtered_df = df[df[numeric_field] > threshold] if numeric_field else df.copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field
if numeric_field and not filtered_df.empty:
    normalized_field = f"{numeric_field}_normalized"
    filtered_df[normalized_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, normalized_field]].head())

# Group by group_field and compute mean of all numeric columns
if group_field and group_field in filtered_df.columns and not filtered_df.empty:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}: (showing top entries)")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we plot the distribution of the selected numeric field and show a barplot of the grouped averages if grouping was performed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the selected numeric field
if numeric_field and not df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

# Bar plot for grouped mean values
if group_field and numeric_field and 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10, 5))
    grouped_df[numeric_field].sort_values(ascending=False).head(20).plot(kind='bar')
    plt.title(f"Top 20 Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" using `mlcroissant`. We loaded the metadata, examined the available record sets and fields by their `@id`, extracted tabular data, performed basic EDA (filtering, normalization, grouping), and visualized key patterns. These steps serve as a foundation for further scientific or ML analysis of protein abundance and characterization in the dataset.